# Synthetic_Hetero — TPL Heuristic Evaluation (with new MAD-based heuristic)

**8 Models** | **100 Epochs** | **4 Outlier Types × 3 Levels** | **5 Runs**

Models compared:
- Pinball, TPL-3sig, TPL-tau, **TPL-tauMAD (new)**, TPL-sweep, HPB, MAQR, QRF

α for each TPL variant is computed twice:
- **Clean-α**: from residuals of MSE-MLP trained on clean data → used for clean evaluation
- **Contam-α**: recomputed from each contaminated set's residuals → used for that condition

**Synthetic-specific**: ground-truth quantiles available → adds RMSE-vs-true-Q analysis.

Plots saved to `plots/Synthetic_Hetero/`

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from scipy.stats import norm, skewnorm
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression as LR_sk
import warnings, time, os

warnings.filterwarnings("ignore")

DN = "Synthetic_Hetero"
PLOTS_DIR = f"plots/{DN}"
os.makedirs(PLOTS_DIR, exist_ok=True)

SPLIT_SEED = 58
SEEDS = [42, 58, 123, 256, 789]
N_RUNS = len(SEEDS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | Runs: {N_RUNS} | Plots → {PLOTS_DIR}/")

QUANTILES = [
    0.01,
    0.025,
    0.03,
    0.05,
    0.09,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.91,
    0.93,
    0.95,
    0.97,
    0.975,
    0.99,
]
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
CONTAMINATION_LEVELS = [0.02, 0.05, 0.10]
EP = 100
H = 100
LR = 0.01
BS = 32

MODELS = [
    "Pinball",
    "TPL-3sig",
    "TPL-tau",
    "TPL-tauMAD",
    "TPL-sweep",
    "HPB",
    "MAQR",
    "QRF",
]
COLORS = {
    "Pinball": "salmon",
    "TPL-3sig": "steelblue",
    "TPL-tau": "crimson",
    "TPL-tauMAD": "purple",
    "TPL-sweep": "seagreen",
    "HPB": "darkorange",
    "MAQR": "brown",
    "QRF": "gray",
}
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (13, 5), "font.size": 11})

Device: cuda | Runs: 5 | Plots → plots/Synthetic_Hetero/


In [2]:
np.random.seed(SPLIT_SEED)
n = 1000
X = np.random.uniform(0, 2, n)
SIGMA_SCALE = 1.0  # base sigma in heteroscedastic noise


def true_mean(x):
    return x


# heteroscedastic noise: σ(x) = SIGMA_SCALE * x
Y_clean = true_mean(X) + np.random.normal(0, SIGMA_SCALE * X, n)
X_col = X.reshape(-1, 1)

DIM = 1
Xtv, X_test, ytv, y_test_clean = train_test_split(
    X_col, Y_clean, test_size=0.20, random_state=SPLIT_SEED
)
X_train, X_val, y_train_clean, y_val_clean = train_test_split(
    Xtv, ytv, test_size=0.20, random_state=SPLIT_SEED
)
print(f"{DN}: Tr={X_train.shape[0]} Val={X_val.shape[0]} Te={X_test.shape[0]}")


def true_quantile(x, tau):
    # Y = x + N(0, (SIGMA_SCALE * x)^2) → conditional quantile = x + (SIGMA_SCALE * x) * Φ⁻¹(τ)
    return true_mean(x) + (SIGMA_SCALE * x) * norm.ppf(tau)

Synthetic_Hetero: Tr=640 Val=160 Te=200


In [3]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(X_col.ravel(), Y_clean, s=12, alpha=0.6, color="steelblue")
ax.set_title(f"{DN} — Clean Training Data")
ax.set_xlabel("X")
ax.set_ylabel("Y")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/{DN}_Clean_Data_Scatter.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {DN}_Clean_Data_Scatter.png")

Saved: Synthetic_Hetero_Clean_Data_Scatter.png


In [4]:
def inject_outliers(y, frac, method, seed=None):
    rng = np.random.RandomState(seed)
    y = y.copy()
    n = len(y)
    no = max(1, int(frac * n))
    ix = rng.choice(n, no, replace=False)
    mu, sig = y.mean(), y.std()
    if method == "gaussian":
        y[ix] = rng.normal(mu, np.sqrt(5) * sig, no)
    elif method == "multiply":
        y[ix] = y[ix] * rng.choice([-1, 1], no) * 10
    elif method == "uniform":
        y[ix] = rng.uniform(y.min() - 3 * sig, y.max() + 3 * sig, no)
    elif method == "skewed":
        y[ix] = skewnorm.rvs(a=10, loc=mu, scale=3 * sig, size=no, random_state=rng)
    return y


cont_sets = {}
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        cont_sets[(ot, cl)] = inject_outliers(y_train_clean, cl, ot, seed=SPLIT_SEED)
print(f"{len(cont_sets)} contaminated sets")

12 contaminated sets


In [5]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
for ax, ot in zip(axs.ravel(), OUTLIER_TYPES):
    yc = cont_sets[(ot, 0.10)]
    ax.scatter(
        X_train.ravel(),
        y_train_clean,
        s=10,
        alpha=0.4,
        color="steelblue",
        label="Clean",
    )
    diff_mask = yc != y_train_clean
    ax.scatter(
        X_train.ravel()[diff_mask],
        yc[diff_mask],
        s=20,
        alpha=0.8,
        color="red",
        label="Injected outlier",
    )
    ax.set_title(f"{ot.capitalize()} (10% contamination)")
    ax.legend(fontsize=9)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
plt.suptitle(f"{DN} — Outlier Injection Visualization (10%)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(
    f"{PLOTS_DIR}/{DN}_Outlier_Injection_Visualization.png",
    dpi=120,
    bbox_inches="tight",
)
plt.close()
print(f"Saved: {DN}_Outlier_Injection_Visualization.png")

Saved: Synthetic_Hetero_Outlier_Injection_Visualization.png


In [6]:
# Paper Figure: Clean vs Outlier-injected scatter (side-by-side)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_col.ravel(), Y_clean, s=10, alpha=0.5, color='steelblue', label='Data')
axes[0].set_title(f'Scatter Plot of (X, y)', fontsize=12)
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
axes[0].legend(fontsize=9)

yc_plot = cont_sets[('gaussian', 0.10)] if ('gaussian', 0.10) in cont_sets else Y_clean
axes[1].scatter(X_train.ravel(), yc_plot, s=10, alpha=0.5, color='orange', label='Data with Outliers')
axes[1].set_title(f'Scatter Plot of (X, y_outlier)', fontsize=12)
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'sh1.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{PLOTS_DIR}/{DN}_Clean_vs_Outlier_Scatter.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Saved: sh1.png')


Saved: sh1.png


In [7]:
class MLP(nn.Module):
    def __init__(s, d):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d, H), nn.ReLU(), nn.Linear(H, 1))

    def forward(s, x):
        return s.net(x).squeeze(-1)


def pb_loss(p, y, tau):
    e = y - p
    return torch.mean(torch.max(tau * e, (tau - 1) * e))


def tpl_loss(p, y, tau, alpha):
    u = y - p
    return torch.mean(
        torch.where(
            (u >= 0) & (u < alpha * (1 - tau)),
            tau * u,
            torch.where(
                u >= alpha * (1 - tau),
                torch.tensor(tau * (1 - tau) * alpha, device=u.device, dtype=u.dtype),
                torch.where(
                    (u < 0) & (u >= -tau * alpha),
                    (tau - 1) * u,
                    torch.tensor(
                        -tau * (tau - 1) * alpha, device=u.device, dtype=u.dtype
                    ),
                ),
            ),
        )
    )


def hpb_loss(p, y, tau, delta=1.0):
    e = y - p
    ae = torch.abs(e)
    h = torch.where(ae <= delta, 0.5 * e**2 / delta, ae - 0.5 * delta)
    return torch.mean(torch.where(e >= 0, tau, 1 - tau) * h)


def sseed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def tr(X, y, lfn, ep=EP):
    m = MLP(DIM).to(DEVICE)
    o = optim.Adam(m.parameters(), lr=LR)
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True
    )
    m.train()
    for _ in range(ep):
        for xb, yb in dl:
            o.zero_grad()
            lfn(m(xb), yb).backward()
            o.step()
    m.eval()
    return m


def pr(m, X):
    m.eval()
    with torch.no_grad():
        return m(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()


def tr_mse(X, y, ep=EP):
    m = MLP(DIM).to(DEVICE)
    o = optim.Adam(m.parameters(), lr=LR)
    L = nn.MSELoss()
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True
    )
    m.train()
    for _ in range(ep):
        for xb, yb in dl:
            o.zero_grad()
            L(m(xb), yb).backward()
            o.step()
    m.eval()
    return m


print("Models ready.")

Models ready.


In [8]:
class MAQR:
    def __init__(s, bfn, k=50, mx=2000):
        s.bfn = bfn
        s.k = k
        s.mx = mx

    def fit(s, X, y):
        s.res = y.ravel() - s.bfn(X).ravel()
        n = len(X)
        if n > s.mx:
            ix = np.random.choice(n, s.mx, replace=False)
            Xs, rs = X[ix], s.res[ix]
        else:
            Xs, rs = X, s.res
        s.nn = NearestNeighbors(n_neighbors=min(s.k, len(Xs)))
        s.nn.fit(Xs)
        Dx, Dp, Dr = [], [], []
        for i in range(len(Xs)):
            _, idx = s.nn.kneighbors(Xs[i : i + 1])
            nr = rs[idx[0]]
            pv = rankdata(nr, "ordinal") / (len(nr) + 1)
            for j in range(len(idx[0])):
                Dx.append(Xs[i])
                Dp.append(pv[j])
                Dr.append(nr[j])
        s.lm = LR_sk()
        s.lm.fit(np.hstack([np.array(Dx), np.array(Dp).reshape(-1, 1)]), np.array(Dr))

    def predict(s, X, qs):
        bp = s.bfn(X).ravel()
        R = np.zeros((len(X), len(qs)))
        for i, t in enumerate(qs):
            R[:, i] = bp + s.lm.predict(np.hstack([X, np.full((len(X), 1), t)]))
        return R


class QRF:
    def __init__(s, rs=42):
        s.rf = RandomForestRegressor(
            n_estimators=100, random_state=rs, min_samples_leaf=5
        )

    def fit(s, X, y):
        s.rf.fit(X, y.ravel())

    def predict(s, X, tau):
        return np.percentile(
            [t.predict(X) for t in s.rf.estimators_], tau * 100, axis=0
        )


print("MAQR/QRF ready.")

MAQR/QRF ready.


In [9]:
sseed(SPLIT_SEED)
mse_m = tr_mse(X_train, y_train_clean)
res_clean = y_train_clean - pr(mse_m, X_train)

RES_STD_CLEAN = np.std(res_clean)
MAD_CLEAN = np.median(np.abs(res_clean - np.median(res_clean)))
SIGMA_MAD_CLEAN = 1.4826 * MAD_CLEAN

A3_CLEAN = 3 * RES_STD_CLEAN

SG = [5, 10, 20, 40, 60, 80, 100, 150, 200, 250, 300]
bse, ASWEEP = 1e9, 40
for a in SG:
    pv = {}
    for t in [0.05, 0.25, 0.50, 0.75, 0.95]:
        pv[t] = pr(
            tr(X_train, y_train_clean, lambda p, y, t=t, a=a: tpl_loss(p, y, t, a)),
            X_val,
        )
    e = np.mean([abs(np.mean(y_val_clean <= pv[t]) - t) for t in pv])
    if e < bse:
        bse, ASWEEP = e, a

print(f"CLEAN-DATA α values:")
print(f"  res_std={RES_STD_CLEAN:.4f}   →  α_3sig={A3_CLEAN:.4f}")
print(f"  1.4826·MAD={SIGMA_MAD_CLEAN:.4f}")
print(f"  TPL-tau:    α(τ) = 3 × {RES_STD_CLEAN:.4f} / min(τ, 1−τ)")
print(f"  TPL-tauMAD: α(τ) = 3 × {SIGMA_MAD_CLEAN:.4f} / min(τ, 1−τ)")
print(f"  α_sweep={ASWEEP}")


def ece(yt, yp, t):
    return abs(np.mean(yt <= yp) - t)

CLEAN-DATA α values:
  res_std=1.0776   →  α_3sig=3.2328
  1.4826·MAD=0.7167
  TPL-tau:    α(τ) = 3 × 1.0776 / min(τ, 1−τ)
  TPL-tauMAD: α(τ) = 3 × 0.7167 / min(τ, 1−τ)
  α_sweep=100


In [10]:
def train_all(Xtr, ytr, Xev, a3, sigma_std, sigma_mad, asw, seed):
    sseed(seed)
    R = {m: {} for m in MODELS}
    for tau in QUANTILES:
        R["Pinball"][tau] = pr(tr(Xtr, ytr, lambda p, y, t=tau: pb_loss(p, y, t)), Xev)
        R["TPL-3sig"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=a3: tpl_loss(p, y, t, a)), Xev
        )
        at_std = 3 * sigma_std / min(tau, 1 - tau)
        R["TPL-tau"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=at_std: tpl_loss(p, y, t, a)), Xev
        )
        at_mad = 3 * sigma_mad / min(tau, 1 - tau)
        R["TPL-tauMAD"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=at_mad: tpl_loss(p, y, t, a)), Xev
        )
        R["TPL-sweep"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=asw: tpl_loss(p, y, t, a)), Xev
        )
        R["HPB"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau: hpb_loss(p, y, t, 1.0)), Xev
        )
    mm = tr_mse(Xtr, ytr)
    mq = MAQR(lambda x: pr(mm, x), k=min(50, len(Xtr) // 5), mx=2000)
    mq.fit(Xtr, ytr)
    mp = mq.predict(Xev, QUANTILES)
    for i, t in enumerate(QUANTILES):
        R["MAQR"][t] = mp[:, i]
    qrf = QRF(rs=seed)
    qrf.fit(Xtr, ytr)
    for t in QUANTILES:
        R["QRF"][t] = qrf.predict(Xev, t)
    return R

In [11]:
Ce = {m: {t: [] for t in QUANTILES} for m in MODELS}
Oe = {
    m: {
        (o, c): {t: [] for t in QUANTILES}
        for o in OUTLIER_TYPES
        for c in CONTAMINATION_LEVELS
    }
    for m in MODELS
}
Or = {
    m: {
        (o, c): {t: [] for t in QUANTILES}
        for o in OUTLIER_TYPES
        for c in CONTAMINATION_LEVELS
    }
    for m in MODELS
}
sc = None  # store run-1 clean predictions for true-Q analysis
sc_out = None  # store run-1 gaussian-10% predictions for paper plots

t0 = time.time()
for ri, seed in enumerate(SEEDS):
    print(f"\nRUN {ri+1}/{N_RUNS} (seed={seed})")
    cr = train_all(
        X_train,
        y_train_clean,
        X_test,
        A3_CLEAN,
        RES_STD_CLEAN,
        SIGMA_MAD_CLEAN,
        ASWEEP,
        seed,
    )
    if ri == 0:
        sc = cr
    for m in MODELS:
        for t in QUANTILES:
            Ce[m][t].append(ece(y_test_clean, cr[m][t], t))

    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            yc = cont_sets[(ot, cl)]
            sseed(seed)
            mc = tr_mse(X_train, yc)
            res_c = yc - pr(mc, X_train)
            sigma_std_c = np.std(res_c)
            sigma_mad_c = 1.4826 * np.median(np.abs(res_c - np.median(res_c)))
            a3_c = 3 * sigma_std_c
            orr = train_all(
                X_train, yc, X_test, a3_c, sigma_std_c, sigma_mad_c, ASWEEP, seed
            )
            if ri == 0 and ot == 'gaussian' and cl == 0.10:
                sc_out = orr  # save for paper plots
            for m in MODELS:
                for t in QUANTILES:
                    Or[m][(ot, cl)][t].append(
                        np.sqrt(mean_squared_error(cr[m][t], orr[m][t]))
                    )
                    Oe[m][(ot, cl)][t].append(ece(y_test_clean, orr[m][t], t))
    print(f"  Done ({(time.time()-t0)/60:.1f}m)")
print(f"\n✓ {N_RUNS} runs in {(time.time()-t0)/60:.1f}m")


RUN 1/5 (seed=42)
  Done (70.6m)

RUN 2/5 (seed=58)
  Done (135.9m)

RUN 3/5 (seed=123)
  Done (206.0m)

RUN 4/5 (seed=256)
  Done (272.1m)

RUN 5/5 (seed=789)
  Done (334.6m)

✓ 5 runs in 334.6m


In [12]:
# ─── Paper Quantile Curve Plots (Figures 2-4) ───
# Uses sc_out (gaussian 10% contaminated predictions, run 1) and sc (clean, run 1)
# Generates: ph1.png, mh1.png, th1.png

assert sc_out is not None, 'sc_out not saved — rerun Cell 10'

# Sort test points by X for smooth curves
sort_idx = np.argsort(X_test.ravel())
X_sorted = X_test.ravel()[sort_idx]

# True quantile curves on sorted grid
tq_lo = true_quantile(X_sorted, 0.025)
tq_hi = true_quantile(X_sorted, 0.975)

# Contaminated training data for background scatter
yc_gauss10 = cont_sets[('gaussian', 0.10)]

paper_models = [
    ('Pinball',    'ph1.png', 'Quantile Regression using pinball loss Predictions'),
    ('MAQR',       'mh1.png', 'MAQR Quantile Regression Predictions with Neural Network Base Model'),
    ('TPL-tauMAD', 'th1.png', 'Quantile Regression using Trimmed pinball loss Predictions'),
]

for model_key, out_fname, title in paper_models:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Background: contaminated training scatter
    ax.scatter(X_train.ravel(), yc_gauss10, s=8, alpha=0.35, color='orange', label='Training Data', zorder=1)
    
    # Predicted quantile curves (from contaminated model)
    pred_lo = sc_out[model_key][0.025][sort_idx]
    pred_hi = sc_out[model_key][0.975][sort_idx]
    
    ax.plot(X_sorted, pred_lo, '--', color='blue', lw=1.5, label=f'predicted quantile at tau = 0.025', zorder=3)
    ax.plot(X_sorted, tq_lo,  '-',  color='red',  lw=1.5, label=f'true quantile at tau = 0.025', zorder=2)
    ax.plot(X_sorted, pred_hi, '--', color='blue', lw=1.5, label=f'predicted quantile at tau = 0.975', zorder=3)
    ax.plot(X_sorted, tq_hi,  '-',  color='red',  lw=1.5, label=f'true quantile at tau = 0.975', zorder=2)
    
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, ls='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_fname, dpi=150, bbox_inches='tight')
    plt.savefig(f'{PLOTS_DIR}/{DN}_Paper_{model_key}_QuantileCurves.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out_fname}')

print(f'\n✓ Paper quantile curve plots generated')


Saved: ph1.png
Saved: mh1.png
Saved: th1.png

✓ Paper quantile curve plots generated


In [13]:
x = list(range(len(QUANTILES)))
xt = [str(q) for q in QUANTILES]
print("Plot setup ready.")

Plot setup ready.


In [14]:
for m in MODELS:
    fig, ax = plt.subplots(figsize=(13, 5))
    mc = [np.mean(Ce[m][t]) for t in QUANTILES]
    sc_ = [np.std(Ce[m][t]) for t in QUANTILES]
    ax.errorbar(
        x, mc, yerr=sc_, fmt="o-", color=COLORS[m], lw=2, markersize=5, capsize=3
    )
    ax.set_xticks(x)
    ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{DN} — Clean ECE per Quantile: {m}", fontsize=12)
    ax.set_xlabel("Quantile τ")
    ax.set_ylabel("Empirical Calibration Error")
    ax.grid(True, ls="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/{DN}_Clean_ECE_{m}.png", dpi=120, bbox_inches="tight")
    plt.close()
print(f"Saved {len(MODELS)} clean ECE per-model plots")

Saved 8 clean ECE per-model plots


In [15]:
fig, ax = plt.subplots(figsize=(14, 6))
for m in MODELS:
    mc = [np.mean(Ce[m][t]) for t in QUANTILES]
    ax.plot(x, mc, "o-", color=COLORS[m], lw=2, markersize=4, label=m)
ax.set_xticks(x)
ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
ax.set_title(f"{DN} — Clean ECE: All Models", fontsize=12)
ax.legend(fontsize=9)
ax.set_xlabel("Quantile τ")
ax.set_ylabel("Empirical Calibration Error")
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/{DN}_Clean_ECE_All_Models.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {DN}_Clean_ECE_All_Models.png")

Saved: Synthetic_Hetero_Clean_ECE_All_Models.png


In [16]:
n_plots = 0
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        for m in MODELS:
            fig, ax = plt.subplots(figsize=(13, 5))
            mn = [np.mean(Or[m][(ot, cl)][t]) for t in QUANTILES]
            sd = [np.std(Or[m][(ot, cl)][t]) for t in QUANTILES]
            ax.errorbar(
                x, mn, yerr=sd, fmt="o-", color=COLORS[m], lw=2, markersize=5, capsize=3
            )
            ax.set_xticks(x)
            ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
            ax.set_title(
                f"{DN} — RMSE per Quantile: {m} ({ot.capitalize()} {int(cl*100)}% contamination)",
                fontsize=11,
            )
            ax.set_xlabel("Quantile τ")
            ax.set_ylabel("RMSE (clean vs outlier predictions)")
            ax.grid(True, ls="--", alpha=0.5)
            plt.tight_layout()
            fname = f"{PLOTS_DIR}/{DN}_RMSE_{m}_{ot.capitalize()}_{int(cl*100)}pct.png"
            plt.savefig(fname, dpi=120, bbox_inches="tight")
            plt.close()
            n_plots += 1
print(f"Saved {n_plots} per-model RMSE plots")

Saved 96 per-model RMSE plots


In [17]:
n_plots = 0
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        for m in MODELS:
            fig, ax = plt.subplots(figsize=(13, 5))
            mc = [np.mean(Ce[m][t]) for t in QUANTILES]
            sc_ = [np.std(Ce[m][t]) for t in QUANTILES]
            mo = [np.mean(Oe[m][(ot, cl)][t]) for t in QUANTILES]
            so_ = [np.std(Oe[m][(ot, cl)][t]) for t in QUANTILES]
            ax.errorbar(
                x,
                mc,
                yerr=sc_,
                fmt="o-",
                color=COLORS[m],
                lw=2,
                markersize=5,
                capsize=3,
                label="Clean",
            )
            ax.errorbar(
                x,
                mo,
                yerr=so_,
                fmt="s--",
                color=COLORS[m],
                lw=1.5,
                markersize=5,
                capsize=3,
                alpha=0.7,
                label=f"{ot.capitalize()} {int(cl*100)}%",
            )
            ax.set_xticks(x)
            ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
            ax.set_title(
                f"{DN} — ECE Clean vs Contaminated: {m} ({ot.capitalize()} {int(cl*100)}%)",
                fontsize=11,
            )
            ax.legend()
            ax.set_xlabel("Quantile τ")
            ax.set_ylabel("Empirical Calibration Error")
            ax.grid(True, ls="--", alpha=0.5)
            plt.tight_layout()
            fname = f"{PLOTS_DIR}/{DN}_ECE_CleanVs_{m}_{ot.capitalize()}_{int(cl*100)}pct.png"
            plt.savefig(fname, dpi=120, bbox_inches="tight")
            plt.close()
            n_plots += 1
print(f"Saved {n_plots} per-model ECE clean-vs-contaminated plots")

Saved 96 per-model ECE clean-vs-contaminated plots


In [18]:
n_plots = 0
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        fig, ax = plt.subplots(figsize=(14, 6))
        for m in MODELS:
            mn = [np.mean(Or[m][(ot, cl)][t]) for t in QUANTILES]
            ax.plot(x, mn, "o-", color=COLORS[m], lw=2, markersize=4, label=m)
        ax.set_xticks(x)
        ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
        ax.set_title(
            f"{DN} — RMSE All Models ({ot.capitalize()} {int(cl*100)}% contamination)",
            fontsize=12,
        )
        ax.legend(fontsize=9)
        ax.set_xlabel("Quantile τ")
        ax.set_ylabel("RMSE (clean vs outlier predictions)")
        ax.grid(True, ls="--", alpha=0.5)
        plt.tight_layout()
        fname = (
            f"{PLOTS_DIR}/{DN}_RMSE_All_Models_{ot.capitalize()}_{int(cl*100)}pct.png"
        )
        plt.savefig(fname, dpi=120, bbox_inches="tight")
        plt.close()
        n_plots += 1
print(f"Saved {n_plots} all-models RMSE overlay plots")

Saved 12 all-models RMSE overlay plots


In [19]:
# RMSE vs TRUE QUANTILES (synthetic only)
true_q = {t: true_quantile(X_test.ravel(), t) for t in QUANTILES}

print(f"\n=== RMSE vs True Quantiles (Run 1, Clean) ===")
trueQ_rows = []
for m in MODELS:
    r = {"Model": m}
    for t in QUANTILES:
        r[t] = round(np.sqrt(mean_squared_error(true_q[t], sc[m][t])), 4)
    r["Avg"] = round(
        np.mean([np.sqrt(mean_squared_error(true_q[t], sc[m][t])) for t in QUANTILES]),
        4,
    )
    trueQ_rows.append(r)
trueQ_df = pd.DataFrame(trueQ_rows)
print(trueQ_df.to_string(index=False))

# Overlay plot
fig, ax = plt.subplots(figsize=(14, 6))
for m in MODELS:
    rv = [np.sqrt(mean_squared_error(true_q[t], sc[m][t])) for t in QUANTILES]
    ax.plot(x, rv, "o-", color=COLORS[m], lw=2, markersize=4, label=m)
ax.set_xticks(x)
ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
ax.set_title(f"{DN} — RMSE vs True Quantiles (All Models, Clean)", fontsize=12)
ax.legend(fontsize=9)
ax.set_xlabel("Quantile τ")
ax.set_ylabel("RMSE (predicted vs true quantile)")
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(
    f"{PLOTS_DIR}/{DN}_RMSE_vs_TrueQuantiles_All_Models.png",
    dpi=120,
    bbox_inches="tight",
)
plt.close()
print(f"Saved: {DN}_RMSE_vs_TrueQuantiles_All_Models.png")


=== RMSE vs True Quantiles (Run 1, Clean) ===
     Model   0.01  0.025   0.03   0.05   0.09    0.1    0.2    0.3    0.4    0.5    0.6    0.7    0.8    0.9   0.91   0.93   0.95   0.97  0.975   0.99    Avg
   Pinball 0.3562 0.1576 0.1682 0.1936 0.1331 0.1126 0.1067 0.0691 0.1864 0.0904 0.1648 0.1746 0.2185 0.1065 0.1077 0.1406 0.4059 0.2442 0.4693 0.2768 0.1941
  TPL-3sig 1.5758 1.1817 1.1005 1.4671 1.3345 1.0061 0.8156 0.4513 0.2906 0.1089 0.2152 0.2770 0.5154 1.0162 1.1215 1.1590 1.1220 1.4220 1.2555 7.5846 1.2510
   TPL-tau 0.3035 0.1953 0.1715 0.1221 0.1719 0.1580 0.1240 0.0682 0.1330 0.1338 0.0863 0.1466 0.1610 0.2167 0.2103 0.1812 0.1722 0.3414 0.1649 0.3271 0.1795
TPL-tauMAD 0.2542 0.1778 0.1375 0.1828 0.1502 0.1091 0.1311 0.1113 0.1545 0.1023 0.1062 0.1725 0.2678 0.1819 0.3752 0.2773 0.2050 0.5022 0.2124 0.2228 0.2017
 TPL-sweep 0.2083 0.1956 0.1052 0.1663 0.1074 0.1213 0.1097 0.0857 0.1748 0.0889 0.1285 0.1334 0.2327 0.0795 0.1490 0.3196 0.1628 0.2427 0.2424 0.2357 0.1645
     

In [20]:
for m in MODELS:
    fig, ax = plt.subplots(figsize=(13, 5))
    rv = [np.sqrt(mean_squared_error(true_q[t], sc[m][t])) for t in QUANTILES]
    ax.plot(x, rv, "o-", color=COLORS[m], lw=2, markersize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{DN} — RMSE vs True Quantiles: {m}", fontsize=12)
    ax.set_xlabel("Quantile τ")
    ax.set_ylabel("RMSE (predicted vs true quantile)")
    ax.grid(True, ls="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(
        f"{PLOTS_DIR}/{DN}_RMSE_vs_TrueQuantiles_{m}.png", dpi=120, bbox_inches="tight"
    )
    plt.close()
print(f"Saved {len(MODELS)} per-model true-Q RMSE plots")

Saved 8 per-model true-Q RMSE plots


In [21]:
# Paper-style tables — 'RMSE under different outlier percentages'
# Replicates paper Tables 2 & 3 layout: rows=quantiles, columns=contamination levels.
# One table per (method, outlier_type).
paper_tables = {}  # paper_tables[(method, outlier_type)] = DataFrame
for m in MODELS:
    for ot in OUTLIER_TYPES:
        rows = []
        for t in QUANTILES:
            row = {"Quantile τ": t}
            for cl in CONTAMINATION_LEVELS:
                row[f"{int(cl*100)}% Outliers"] = round(np.mean(Or[m][(ot, cl)][t]), 4)
            rows.append(row)
        paper_tables[(m, ot)] = pd.DataFrame(rows)

# Print example: Pinball + Gaussian (mirrors paper Table 2)
print("Example — Pinball under Gaussian contamination:")
print(paper_tables[("Pinball", "gaussian")].to_string(index=False))
print(
    f"\n{len(paper_tables)} paper-style RMSE tables built ({len(MODELS)} methods × {len(OUTLIER_TYPES)} outlier types)"
)

Example — Pinball under Gaussian contamination:
 Quantile τ  2% Outliers  5% Outliers  10% Outliers
      0.010       0.2308       0.3330        0.5618
      0.025       0.1913       0.1902        0.4065
      0.030       0.0865       0.1183        0.2560
      0.050       0.0737       0.1625        0.1609
      0.090       0.0295       0.0912        0.1119
      0.100       0.0475       0.1102        0.1059
      0.200       0.0207       0.0289        0.0948
      0.300       0.0208       0.0283        0.0353
      0.400       0.0322       0.0487        0.0409
      0.500       0.0362       0.0254        0.0449
      0.600       0.0311       0.0268        0.0600
      0.700       0.0370       0.0454        0.0731
      0.800       0.0503       0.0921        0.1552
      0.900       0.0816       0.1528        0.3073
      0.910       0.1163       0.2237        0.2700
      0.930       0.1206       0.1722        0.4201
      0.950       0.2233       0.2637        0.5492
      0.970     

In [22]:
print(f"\nSUMMARY — {DN}")
print(
    f"Clean-α: α_3sig={A3_CLEAN:.4f} | σ_std={RES_STD_CLEAN:.4f} | σ_MAD={SIGMA_MAD_CLEAN:.4f} | α_sweep={ASWEEP}"
)
rows = []
for m in MODELS:
    r = {
        "Model": m,
        "ECE_clean": round(np.mean([np.mean(Ce[m][t]) for t in QUANTILES]), 4),
        "RMSE_vs_TrueQ_clean": round(
            np.mean(
                [np.sqrt(mean_squared_error(true_q[t], sc[m][t])) for t in QUANTILES]
            ),
            4,
        ),
    }
    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            k = (ot, cl)
            r[f"RMSE_{ot}_{int(cl*100)}"] = round(
                np.mean([np.mean(Or[m][k][t]) for t in QUANTILES]), 4
            )
            r[f"ECE_{ot}_{int(cl*100)}"] = round(
                np.mean([np.mean(Oe[m][k][t]) for t in QUANTILES]), 4
            )
    rows.append(r)
sdf = pd.DataFrame(rows)
print(sdf.to_string(index=False))


SUMMARY — Synthetic_Hetero
Clean-α: α_3sig=3.2328 | σ_std=1.0776 | σ_MAD=0.7167 | α_sweep=100
     Model  ECE_clean  RMSE_vs_TrueQ_clean  RMSE_gaussian_2  ECE_gaussian_2  RMSE_gaussian_5  ECE_gaussian_5  RMSE_gaussian_10  ECE_gaussian_10  RMSE_multiply_2  ECE_multiply_2  RMSE_multiply_5  ECE_multiply_5  RMSE_multiply_10  ECE_multiply_10  RMSE_uniform_2  ECE_uniform_2  RMSE_uniform_5  ECE_uniform_5  RMSE_uniform_10  ECE_uniform_10  RMSE_skewed_2  ECE_skewed_2  RMSE_skewed_5  ECE_skewed_5  RMSE_skewed_10  ECE_skewed_10
   Pinball     0.0254               0.1941           0.1071          0.0283           0.2011          0.0297            0.4057           0.0328           0.5573          0.0293           1.6795          0.0337            4.0172           0.0402          0.1489         0.0272          0.4593         0.0314           1.2547          0.0378         0.1494        0.0250         0.4022        0.0278          0.7264         0.0329
  TPL-3sig     0.1497               1.2510     

In [23]:
import openpyxl

ep = f"{DN}_results.xlsx"
with pd.ExcelWriter(ep, engine="openpyxl") as w:
    pd.DataFrame(
        {
            "Method": ["TPL-3sig", "TPL-tau", "TPL-tauMAD", "TPL-sweep"],
            "Alpha (clean-data)": [
                round(A3_CLEAN, 4),
                f"3×{RES_STD_CLEAN:.4f}/min(τ,1-τ)",
                f"3×{SIGMA_MAD_CLEAN:.4f}/min(τ,1-τ)  [1.4826·MAD]",
                ASWEEP,
            ],
        }
    ).to_excel(w, sheet_name="Alpha", index=False)

    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            k = (ot, cl)
            rows = []
            for m in MODELS:
                r = {"Model": m}
                for t in QUANTILES:
                    r[t] = round(np.mean(Or[m][k][t]), 4)
                r["Avg"] = round(np.mean([np.mean(Or[m][k][t]) for t in QUANTILES]), 4)
                rows.append(r)
            pd.DataFrame(rows).to_excel(
                w, sheet_name=f"RMSE_{ot}_{int(cl*100)}"[:31], index=False
            )

    rows = []
    for m in MODELS:
        r = {"Model": m}
        for t in QUANTILES:
            r[t] = round(np.mean(Ce[m][t]), 4)
        r["Avg"] = round(np.mean([np.mean(Ce[m][t]) for t in QUANTILES]), 4)
        rows.append(r)
    pd.DataFrame(rows).to_excel(w, sheet_name="ECE_Clean", index=False)

    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            k = (ot, cl)
            rows = []
            for m in MODELS:
                r = {"Model": m}
                for t in QUANTILES:
                    r[t] = round(np.mean(Oe[m][k][t]), 4)
                r["Avg"] = round(np.mean([np.mean(Oe[m][k][t]) for t in QUANTILES]), 4)
                rows.append(r)
            pd.DataFrame(rows).to_excel(
                w, sheet_name=f"ECE_{ot}_{int(cl*100)}"[:31], index=False
            )

    # RMSE vs True Q
    trueQ_df.to_excel(w, sheet_name="RMSE_TrueQ", index=False)

    # Paper-style tables, one sheet per (method, outlier_type)
    for (m, ot), df in paper_tables.items():
        sn = f"Paper_{m}_{ot}"[:31]
        df.to_excel(w, sheet_name=sn, index=False)

    sdf.to_excel(w, sheet_name="Summary", index=False)
print(f"✓ {ep}")

✓ Synthetic_Hetero_results.xlsx


## Notes — Synthetic_Hetero

- **TPL-3sig**: α = 3 × std(residuals) (single value)
- **TPL-tau**: α(τ) = 3 × std(residuals) / min(τ, 1−τ)
- **TPL-tauMAD (new)**: α(τ) = 3 × 1.4826·MAD(residuals) / min(τ, 1−τ)
- **TPL-sweep**: α chosen by minimizing validation ECE on clean data
- Clean-α: from clean residuals. Contamination-α: recomputed per condition.
- Paper-style RMSE tables: one per (method, outlier_type), rows=τ, cols=2%/5%/10%
- Synthetic exclusive: RMSE-vs-True-Quantiles analysis using analytic inverse CDF
- 100 epochs, 5 seeds, all plots saved to `plots/Synthetic_Hetero/`


In [24]:
# ─── Additional Paper Plots (run while kernel is alive) ───
import numpy as np

paper_methods = ['Pinball', 'TPL-tauMAD', 'MAQR', 'QRF']
paper_labels  = ['Pinball', 'TPL',        'MAQR', 'QRF']
paper_colors  = ['#e41a1c', '#984ea3',    '#377eb8', '#4daf4a']

xt = [str(q) for q in QUANTILES]
xi = list(range(len(QUANTILES)))

# ── Plot 1: RMSE per quantile under MULTIPLY 10% ──
fig, ax = plt.subplots(figsize=(14, 6))
for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
    vals = [np.mean(Or[m][('multiply', 0.10)][t]) for t in QUANTILES]
    ax.plot(xi, vals, 'o-', color=col, lw=2, markersize=4, label=lab)
ax.set_xticks(xi)
ax.set_xticklabels(xt, rotation=45, ha='right', fontsize=8)
ax.set_title(f'{DN} — RMSE per Quantile (Multiply 10% Contamination)', fontsize=13)
ax.set_xlabel('Quantile τ'); ax.set_ylabel('RMSE (clean vs contaminated)')
ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{DN}_RMSE_per_Quantile_Multiply10.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {DN}_RMSE_per_Quantile_Multiply10.png')

# ── Plot 2: RMSE per quantile under GAUSSIAN 10% ──
fig, ax = plt.subplots(figsize=(14, 6))
for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
    vals = [np.mean(Or[m][('gaussian', 0.10)][t]) for t in QUANTILES]
    ax.plot(xi, vals, 'o-', color=col, lw=2, markersize=4, label=lab)
ax.set_xticks(xi)
ax.set_xticklabels(xt, rotation=45, ha='right', fontsize=8)
ax.set_title(f'{DN} — RMSE per Quantile (Gaussian 10% Contamination)', fontsize=13)
ax.set_xlabel('Quantile τ'); ax.set_ylabel('RMSE (clean vs contaminated)')
ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{DN}_RMSE_per_Quantile_Gaussian10.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {DN}_RMSE_per_Quantile_Gaussian10.png')

# ── Plot 3: RMSE vs True Quantiles (clean — adaptiveness proof) ──
true_q = {t: true_quantile(X_test.ravel(), t) for t in QUANTILES}
fig, ax = plt.subplots(figsize=(14, 6))
for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
    vals = [np.sqrt(np.mean((true_q[t] - sc[m][t])**2)) for t in QUANTILES]
    ax.plot(xi, vals, 'o-', color=col, lw=2, markersize=4, label=lab)
ax.set_xticks(xi)
ax.set_xticklabels(xt, rotation=45, ha='right', fontsize=8)
ax.set_title(f'{DN} — RMSE vs True Quantiles (Clean Data)', fontsize=13)
ax.set_xlabel('Quantile τ'); ax.set_ylabel('RMSE (predicted vs true quantile)')
ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{DN}_RMSE_vs_TrueQ_Paper.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {DN}_RMSE_vs_TrueQ_Paper.png')

# ── Plot 4: ECE per quantile on clean data ──
fig, ax = plt.subplots(figsize=(14, 6))
for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
    vals = [np.mean(Ce[m][t]) for t in QUANTILES]
    ax.plot(xi, vals, 'o-', color=col, lw=2, markersize=4, label=lab)
ax.set_xticks(xi)
ax.set_xticklabels(xt, rotation=45, ha='right', fontsize=8)
ax.set_title(f'{DN} — Clean ECE per Quantile (All Models)', fontsize=13)
ax.set_xlabel('Quantile τ'); ax.set_ylabel('ECE')
ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{DN}_Clean_ECE_Paper.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {DN}_Clean_ECE_Paper.png')

print('\n✓ All 4 additional paper plots generated')

Saved: Synthetic_Hetero_RMSE_per_Quantile_Multiply10.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_Gaussian10.png
Saved: Synthetic_Hetero_RMSE_vs_TrueQ_Paper.png
Saved: Synthetic_Hetero_Clean_ECE_Paper.png

✓ All 4 additional paper plots generated


In [25]:
# ─── RMSE per Quantile for ALL contamination types ───
paper_methods = ['Pinball', 'TPL-tauMAD', 'MAQR', 'QRF']
paper_labels  = ['Pinball', 'TPL',        'MAQR', 'QRF']
paper_colors  = ['#e41a1c', '#984ea3',    '#377eb8', '#4daf4a']

xt = [str(q) for q in QUANTILES]
xi = list(range(len(QUANTILES)))

for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        fig, ax = plt.subplots(figsize=(14, 6))
        for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
            vals = [np.mean(Or[m][(ot, cl)][t]) for t in QUANTILES]
            ax.plot(xi, vals, 'o-', color=col, lw=2, markersize=4, label=lab)
        ax.set_xticks(xi)
        ax.set_xticklabels(xt, rotation=45, ha='right', fontsize=8)
        pct = int(cl * 100)
        ax.set_title(f'{DN} — RMSE per Quantile ({ot.capitalize()} {pct}% Contamination)', fontsize=13)
        ax.set_xlabel('Quantile τ'); ax.set_ylabel('RMSE (clean vs contaminated)')
        ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(f'{PLOTS_DIR}/{DN}_RMSE_per_Quantile_{ot}_{pct}.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Saved: {DN}_RMSE_per_Quantile_{ot}_{pct}.png')

print(f'\n✓ {len(OUTLIER_TYPES) * len(CONTAMINATION_LEVELS)} contamination RMSE plots generated')

Saved: Synthetic_Hetero_RMSE_per_Quantile_gaussian_2.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_gaussian_5.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_gaussian_10.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_multiply_2.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_multiply_5.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_multiply_10.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_uniform_2.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_uniform_5.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_uniform_10.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_skewed_2.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_skewed_5.png
Saved: Synthetic_Hetero_RMSE_per_Quantile_skewed_10.png

✓ 12 contamination RMSE plots generated


In [26]:
# ─── ECE per Quantile for ALL contamination types ───
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        fig, ax = plt.subplots(figsize=(14, 6))
        for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
            vals = [np.mean(Oe[m][(ot, cl)][t]) for t in QUANTILES]
            ax.plot(xi, vals, 'o-', color=col, lw=2, markersize=4, label=lab)
        ax.set_xticks(xi)
        ax.set_xticklabels(xt, rotation=45, ha='right', fontsize=8)
        pct = int(cl * 100)
        ax.set_title(f'{DN} — ECE per Quantile ({ot.capitalize()} {pct}% Contamination)', fontsize=13)
        ax.set_xlabel('Quantile τ'); ax.set_ylabel('ECE')
        ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(f'{PLOTS_DIR}/{DN}_ECE_per_Quantile_{ot}_{pct}.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Saved: {DN}_ECE_per_Quantile_{ot}_{pct}.png')

print(f'\n✓ {len(OUTLIER_TYPES) * len(CONTAMINATION_LEVELS)} contamination ECE plots generated')

Saved: Synthetic_Hetero_ECE_per_Quantile_gaussian_2.png
Saved: Synthetic_Hetero_ECE_per_Quantile_gaussian_5.png
Saved: Synthetic_Hetero_ECE_per_Quantile_gaussian_10.png
Saved: Synthetic_Hetero_ECE_per_Quantile_multiply_2.png
Saved: Synthetic_Hetero_ECE_per_Quantile_multiply_5.png
Saved: Synthetic_Hetero_ECE_per_Quantile_multiply_10.png
Saved: Synthetic_Hetero_ECE_per_Quantile_uniform_2.png
Saved: Synthetic_Hetero_ECE_per_Quantile_uniform_5.png
Saved: Synthetic_Hetero_ECE_per_Quantile_uniform_10.png
Saved: Synthetic_Hetero_ECE_per_Quantile_skewed_2.png
Saved: Synthetic_Hetero_ECE_per_Quantile_skewed_5.png
Saved: Synthetic_Hetero_ECE_per_Quantile_skewed_10.png

✓ 12 contamination ECE plots generated


In [27]:
# ─── Sensitivity: RMSE vs Contamination Level (2% → 5% → 10%) ───
for ot in OUTLIER_TYPES:
    fig, ax = plt.subplots(figsize=(8, 5))
    pcts = [int(cl * 100) for cl in CONTAMINATION_LEVELS]
    for m, lab, col in zip(paper_methods, paper_labels, paper_colors):
        vals = [np.mean([np.mean(Or[m][(ot, cl)][t]) for t in QUANTILES]) for cl in CONTAMINATION_LEVELS]
        ax.plot(pcts, vals, 'o-', color=col, lw=2.5, markersize=8, label=lab)
    ax.set_xticks(pcts)
    ax.set_xticklabels([f'{p}%' for p in pcts])
    ax.set_title(f'{DN} — Avg RMSE vs Contamination Level ({ot.capitalize()})', fontsize=13)
    ax.set_xlabel('Contamination %'); ax.set_ylabel('Average RMSE')
    ax.legend(fontsize=10); ax.grid(True, ls='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/{DN}_Sensitivity_{ot}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {DN}_Sensitivity_{ot}.png')

print('\n✓ 4 sensitivity plots generated')

Saved: Synthetic_Hetero_Sensitivity_gaussian.png
Saved: Synthetic_Hetero_Sensitivity_multiply.png
Saved: Synthetic_Hetero_Sensitivity_uniform.png
Saved: Synthetic_Hetero_Sensitivity_skewed.png

✓ 4 sensitivity plots generated
